In [ ]:
"""
Concepts:

1. Support Vectors
2. Margin Maximization
3. Kernel Trick
4. C Parameter
5. Gamma Parameter
6. Probability Calibration
7. ROC-AUC
8. Hyperparameter Tuning
"""

# =====================================================
# IMPORTS
# =====================================================

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (train_test_split,cross_val_score,GridSearchCV,RandomizedSearchCV,learning_curve)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix,classification_report,roc_curve,precision_recall_curve)
from sklearn.calibration import calibration_curve


In [ ]:
# =====================================================
# STEP 1 : DATA UNDERSTANDING
# =====================================================

data = load_breast_cancer()
df = pd.DataFrame(data.data,columns=data.feature_names)
df["target"] = data.target
print(df.head())
print(df.info())
print(df.describe())
print(df["target"].value_counts())

In [ ]:
# =====================================================
# STEP 2 : EDA
# =====================================================

print(df.isnull().sum())
print(df.duplicated().sum())
print(df["target"].value_counts(normalize=True))

In [ ]:
# =====================================================
# STEP 3 : TRAIN / VALIDATION / TEST
# =====================================================

X = df.drop("target", axis=1)
y = df["target"]

X_temp, X_test, y_temp, y_test = train_test_split(X,y,test_size=0.15,random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp,y_temp,test_size=0.1765,random_state=42)

In [ ]:
# =====================================================
# STEP 4 : PREPROCESSING
# =====================================================

# SVM REQUIRES SCALING

pipe = Pipeline([("scaler",StandardScaler()),("model",SVC(kernel="rbf",C=1.0,gamma="scale",probability=True,random_state=42))])

In [ ]:
# =====================================================
# STEP 5 : BASELINE MODEL
# =====================================================

pipe.fit(X_train,y_train)

In [ ]:
# =====================================================
# STEP 6 : VALIDATION METRICS with base model
# =====================================================

y_pred = pipe.predict(X_val)
val_prob = pipe.predict_proba(X_val)[:,1]
accuracy = accuracy_score(y_val,y_pred)
precision = precision_score(y_val,y_pred)
recall = recall_score(y_val,y_pred)
f1 = f1_score(y_val,y_pred)
auc = roc_auc_score(y_val,y_pred)

In [ ]:
# =====================================================
# STEP 7 : CROSS VALIDATION
# =====================================================

cv_scores = cross_val_score(pipe,X_train,y_train,cv=5,scoring="roc_auc",n_jobs=-1)
print(cv_scores.mean())
print(cv_scores.std())

In [ ]:
# =====================================================
# STEP 8 : HYPERPARAMETER TUNING
# =====================================================

param_grid = {
    "model__C":[0.01,0.1,1,10,100],
    "model__gamma":["scale","auto",0.001,0.01,0.1],
    "model__kernel":["linear","rbf"]}

grid = GridSearchCV(pipe,param_grid,cv=5,scoring="roc_auc",n_jobs=-1)
grid.fit(X_train,y_train)
print(grid.best_params_)
best_model = grid.best_estimator_
# =====================================================
# STEP 8.5 : RANDOMIZED SEARCH
# =====================================================

random_search = RandomizedSearchCV(pipe,param_distributions=param_grid,cv=5,n_iter=15,scoring="roc_auc",random_state=42,n_jobs=-1)
random_search.fit(X_train,y_train)
print(random_search.best_params_)

In [ ]:
# =====================================================
# STEP 9 : VALIDATION RECHECK
# =====================================================

val_pred = best_model.predict(X_val)
val_prob = best_model.predict_proba(X_val)[:,1]
val_auc = roc_auc_score(y_val,val_prob)
print("\nValidation AUC")
print(val_auc)

In [ ]:
# =====================================================
# STEP 10 : TRAIN VS VALIDATION
# =====================================================

train_prob = best_model.predict_proba(X_train)[:,1]
train_auc = roc_auc_score(y_train,train_prob)
print("\nTrain AUC:", train_auc)
print("Validation AUC:", val_auc)

In [ ]:
# =====================================================
# STEP 11 : SVM ANALYSIS
# =====================================================

# ------------------------------------------
# Support Vectors
# ------------------------------------------

svc_model = best_model.named_steps["model"]
len(svc_model.support_)

# ------------------------------------------
# Confusion Matrix
# ------------------------------------------
cm = confusion_matrix(y_val,val_pred)
sns.heatmap(cm,annot=True,fmt='d')
plt.title("Confusion Matrix")
plt.show()

# ------------------------------------------
# Classification Report
# ------------------------------------------
print(classification_report(y_val,val_pred))

# ------------------------------------------
# ROC Curve
# ------------------------------------------

fpr, tpr, _ = roc_curve(y_val,val_prob)
plt.plot(fpr,tpr)
plt.plot([0,1],[0,1])
plt.title("ROC Curve")
plt.show()

# ------------------------------------------
# Precision Recall Curve
# ------------------------------------------

precision_vals, recall_vals, _ = (precision_recall_curve(y_val,val_prob))
plt.plot(recall_vals,precision_vals)
plt.title("PR Curve")
plt.show()

# ------------------------------------------
# Calibration Curve
# ------------------------------------------

prob_true, prob_pred = calibration_curve(y_val,val_prob,n_bins=10)
plt.plot(prob_pred,prob_true,marker='o')
plt.plot([0,1],[0,1])
plt.title("Calibration Curve")
plt.show()

# ------------------------------------------
# Kernel Comparison
# ------------------------------------------

kernels = ["linear","rbf","poly"]
scores = []

for kernel in kernels:
    model = Pipeline([("scaler",StandardScaler()),("svc",SVC(kernel=kernel,probability=True,random_state=42))])

    model.fit(X_train,y_train)
    probs = model.predict_proba(X_val)[:,1]
    score = roc_auc_score(y_val,probs)
    scores.append(score)

plt.bar(kernels,scores)
plt.title("Kernel Comparison")
plt.ylabel("Validation ROC-AUC")
plt.show()

# ------------------------------------------
# C Sensitivity
# ------------------------------------------

c_values = [0.01,0.1,1,10,100]
scores = []

for c in c_values:

    model = Pipeline([("scaler", StandardScaler()),("svc", SVC(C=c,probability=True,random_state=42))])
    model.fit(X_train,y_train)

    probs = model.predict_proba(X_val)[:,1]
    score = roc_auc_score(y_val,probs)
    scores.append(score)

plt.plot(c_values,scores,marker='o')
plt.xscale("log")
plt.title("C Sensitivity")
plt.show()

# ------------------------------------------
# Gamma Sensitivity
# ------------------------------------------

gamma_values = [0.001,0.01,0.1,1]
scores = []

for gamma in gamma_values:
    model = Pipeline([("scaler", StandardScaler()),("svc", SVC(gamma=gamma,probability=True,random_state=42))])

    model.fit(X_train,y_train)
    probs = model.predict_proba(X_val)[:,1]
    score = roc_auc_score(y_val,probs)
    scores.append(score)

plt.plot(gamma_values,scores,marker='o')
plt.xscale("log")
plt.title("Gamma Sensitivity")
plt.show()

# ------------------------------------------
# Learning Curve
# ------------------------------------------

train_sizes, train_scores, val_scores = learning_curve(best_model,X_train,y_train,cv=5,scoring="roc_auc",n_jobs=-1)
plt.plot(train_sizes,np.mean(train_scores, axis=1),label="Train")
plt.plot(train_sizes,np.mean(val_scores, axis=1),label="Validation")
plt.legend()
plt.title("Learning Curve")
plt.show()


In [ ]:
# =====================================================
# STEP 12 : FINAL MODEL
# =====================================================

final_model = best_model


In [ ]:
# =====================================================
# STEP 13 : TEST EVALUATION
# =====================================================

test_pred = final_model.predict(X_test)

test_prob = final_model.predict_proba(X_test)[:,1]

print("\nTEST RESULTS")
print(classification_report(y_test,test_pred))

print("ROC AUC:",roc_auc_score(y_test,test_prob))

In [ ]:
# =====================================================
# STEP 14 : DEPLOYMENT READINESS
# =====================================================

print("\nDeployment Metrics")
print("CV AUC:", cv_scores.mean())
print("Validation AUC:", val_auc)
print("Test AUC:",roc_auc_score(y_test, test_prob))

# =====================================================
# STEP 15 : INTERVIEW QUESTIONS
# =====================================================

"""
1. What is a Support Vector?
2. What is Margin?
3. Why maximize margin?
4. Difference between Logistic Regression and SVM?
5. What is Kernel Trick?
6. Why use RBF kernel?
7. What is Gamma?
8. What is C?
9. What happens if Gamma increases?
10. What happens if C increases?
11. Why is scaling mandatory?
12. Linear Kernel vs RBF Kernel?
13. What is a soft margin
14. What is a hard margin?
15. Why is SVM slow on large datasets?
16. What is class_weight?
17. How do you handle imbalanced data?
18. Why probability=True increases training time?
19. What are support vectors used for?
20. Explain SVM from scratch.
"""